In [9]:
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
load_dotenv() # load all environment variables
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
pinecone_api_key = os.getenv("PINECONE_API_KEY")
hf_token = os.getenv("HF_TOKEN")
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_huggingface import HuggingFaceEmbeddings

In [5]:
index_name = "hybrid-search-langchain"
## initialize pinecone client
pinecone_client = Pinecone(api_key = pinecone_api_key)
if index_name not in pinecone_client.list_indexes().names():
    ## create index if it doesn't exist
    pinecone_client.create_index(
        name=index_name,
        dimension=384, # dimension of the vector embeddings for vector dense search
        metric="dotproduct", # sparse values supported only for dotproduct 
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )
# retriever = PineconeHybridSearchRetriever(
#     pinecone_client=pinecone_client,
#     index_name=index_name,
#     hybrid_search_config={
#         "alpha": 0.5, # balance between vector search and keyword search
#         "k": 10, # number of results to return
#     }
# )

In [7]:
index = pinecone_client.Index(index_name)

e:\00-My_github_repository\GenAiApps\LangChain_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# vector embeddings for dense search
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [11]:
# Sparse values for keyword search
from pinecone_text.sparse import BM25Encoder
bm25_encoder = BM25Encoder().default()

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dell\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dell\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [15]:
sentences = [
    "In 2010, I visited Saudi Arabia for the first time.",
    "In 2020, I visited Paris for the first time.",
    "In 2025, I visited Cairo for the first time.",
]

## applay tfidf on sentences to get sparse values for keyword search
sparse_values = bm25_encoder.fit(sentences)
## store values in json format file
bm25_encoder.dump("sparse_values.json")
## load values from json file
bm25_encoder = BM25Encoder.load(self = bm25_encoder, path = "sparse_values.json")

100%|██████████| 3/3 [00:00<00:00, 2081.54it/s]


In [18]:
# create retriever with pinecone client, index name, and hybrid search config
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=index
)

In [19]:
retriever.add_texts(sentences, ids=["1", "2", "3"])

100%|██████████| 1/1 [00:06<00:00,  6.51s/it]


In [20]:
retriever.invoke("What is the first year I visited a city?")

[Document(metadata={'score': 0.291537076}, page_content='In 2020, I visited Paris for the first time.'),
 Document(metadata={'score': 0.282257348}, page_content='In 2025, I visited Cairo for the first time.'),
 Document(metadata={'score': 0.247082472}, page_content='In 2010, I visited Saudi Arabia for the first time.')]